# E2E 01 -- Builder Graph Smoke Test

End-to-end verification of the Builder Graph using the minimal smoke fixtures.

Checks:
- Neo4j connectivity
- Document loading and chunking
- Triplet extraction
- DDL parsing
- Cypher generation and execution


In [ ]:
from __future__ import annotations
import sys, os
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

os.environ.setdefault('LOG_LEVEL', 'WARNING')
os.environ.setdefault('NEO4J_URI',  'bolt://localhost:7687')
os.environ.setdefault('NEO4J_USER', 'neo4j')
os.environ.setdefault('NEO4J_PASSWORD', 'your_password')
os.environ.setdefault('LMSTUDIO_BASE_URL', 'http://localhost:1234/v1')
os.environ.setdefault('LLM_MODEL_REASONING',  'local-model')
os.environ.setdefault('LLM_MODEL_EXTRACTION', 'local-model')
os.environ.setdefault('LLM_MAX_TOKENS_EXTRACTION', '16384')

fixtures_dir = repo_root / 'tests' / 'fixtures'
doc_paths = [
    fixtures_dir / 'smoke' / 'business_glossary_smoke.txt',
    fixtures_dir / 'smoke' / 'data_dictionary_smoke.txt',
]
ddl_paths = [fixtures_dir / 'smoke' / 'smoke_schema.sql']

print(f'Repository root : {repo_root}')
print(f'Documents       : {[p.name for p in doc_paths]}')
print(f'DDL files       : {[p.name for p in ddl_paths]}')


In [ ]:
from src.graph.neo4j_client import Neo4jClient

with Neo4jClient() as neo4j:
    ok = neo4j.test_connection()

assert ok, 'Neo4j connection failed -- start Neo4j and retry.'
print('Neo4j connection: OK')


In [ ]:
import time
from src.graph.builder_graph import run_builder

print('Running Builder Graph ...')
start = time.time()

final_state = run_builder(
    raw_documents=doc_paths,
    ddl_paths=ddl_paths,
    production=False,
)

elapsed = time.time() - start
print(f'Completed in {elapsed:.1f} s')


In [ ]:
tables    = final_state.get('tables', [])
completed = final_state.get('completed_tables', [])
failed    = final_state.get('cypher_failed', False)

assert len(tables) > 0, f'Expected at least one parsed table, got 0.'
assert not failed,      'Cypher healing failed -- check logs for details.'

print('Builder Graph Assertions')
print('-' * 40)
print(f'Tables parsed     : {len(tables)}')
print(f'Tables completed  : {len(completed)}')
print(f'Cypher failures   : {failed}')
print('-' * 40)
print('All assertions passed.')


In [ ]:
from src.graph.neo4j_client import Neo4jClient

with Neo4jClient() as neo4j:
    node_rows = neo4j.execute_cypher(
        'MATCH (n) RETURN labels(n) AS labels, count(n) AS count ORDER BY count DESC'
    )
    mapping_rows = neo4j.execute_cypher(
        'MATCH (c:BusinessConcept)-[r:MAPPED_TO]->(t:PhysicalTable)'
        ' RETURN c.name AS concept, t.table_name AS table ORDER BY t.table_name'
    )

print('Graph Contents')
print('-' * 40)
for r in node_rows:
    print(f"  {r['labels']}: {r['count']}")

print()
print('Mappings')
print('-' * 40)
for r in mapping_rows:
    print(f"  {r['concept']} -> {r['table']}")
